In [ ]:
#recall: documentation for this specific problem: https://docs.pasqal.com/pulser/tutorials/mwis/$0

In [ ]:
#imports for pulser

import numpy as np
import matplotlib.pyplot as plt
import pulser
import pulser_simulation
from scipy.optimize import minimize
from scipy.spatial.distance import pdist, squareform, euclidean

#imports for interfacing with the rest of the codebase


test_plotting = False

In [ ]:
#useful functions

def evaluate_mapping(new_coords: np.ndarray, Q: np.ndarray, device: pulser.devices.Device):
    """Cost function to minimize. Ideally, the pairwise distances are conserved."""

    new_coords = np.reshape(new_coords, (len(Q), 2))
    # computing the matrix of the distances between all coordinate pairs
    new_Q = squareform(device.interaction_coeff / pdist(new_coords) ** 6) / 4

    return np.linalg.norm(new_Q - Q)

def plot_distribution(C):

    C = dict(sorted(C.items(), key=lambda item: item[1], reverse=True))
    indexes = ["0110"]  # best solution
    color_dict = {key: "r" if key in indexes else "g" for key in C}
    plt.figure(figsize=(12, 6))
    plt.xlabel("bitstrings")
    plt.ylabel("counts")
    plt.bar(C.keys(), C.values(), width=0.5, color=color_dict.values())
    plt.xticks(rotation="vertical")
    plt.show()

In [ ]:
#turn inputs from classical prep into an adjacency matrix of the appropriate form

#placeholder, for now:

Q = np.array(
    [
        [0, 1, 1, 1],
        [1, 2, 0, 0],
        [1, 0, 2, 1],
        [1, 0, 1, 0],
    ]
)

In [ ]:
#first, choose a device.
#the Pasqal documentation uses a "Mock Device" with no "real-world" constraints.
#however, you could imagine we might want to use real-world constraints.
#let's make a Boolean that allows us to switch back and forth

mock_device = True

if mock_device:
  device = pulser.MockDevice

else:
  device = pulser.WeightedAnalogDevice
  #add the necessary channels

# device.print_specs()

In [ ]:
#minimize on cost function

costs = []
np.random.seed(0)
x0 = np.random.random(len(Q) * 2)

res = minimize(
    evaluate_mapping,
    x0,
    args=(~np.eye(Q.shape[0], dtype=bool) * Q, device),
    method="Nelder-Mead",
    tol=1e-6,
    options={"maxiter": 200000, "maxfev": None},
)
coords = np.reshape(res.x, (len(Q), 2))

#place the qubits at coordinates from classical result
qubits = {f"q{i}": coord for (i, coord) in enumerate(coords)}
reg = pulser.Register(qubits)

if test_plotting:
    reg.draw(blockade_radius=device.rydberg_blockade_radius(1.0),draw_graph=True,draw_half_radius=True,)

In [ ]:
#initialize sequence

sequence = pulser.Sequence(reg, device)

#declare Rydberg channel to be able to use it in the sequence

sequence.declare_channel("rydberg_global", "rydberg_global")

#check weights graphically

#grab the weights
node_weights = np.diag(Q)
norm_node_weights = node_weights / np.max(node_weights)

#generate the detuning map
det_map_weights = 1 - norm_node_weights
det_map = reg.define_detuning_map(
    {f"q{i}": det_map_weights[i] for i in range(len(det_map_weights))}
)

if test_plotting:
  det_map.draw(labels=reg.qubit_ids)

#add the local detuning map to apply the verified weights:

sequence.config_detuning_map(det_map, "dmm_0")


In [ ]:
#let's do the pulse shaping for adiabatic evolution.
#first, we need to bound the Rabi frequency so we stay in the adiabatic regime.
#bound the Rabi frequency by that for the closest pair of atoms

distance_non_connected = []
for i in range(1, Q.shape[0]):
    for j in range(i - 1):
        if Q[i, j] == 0:
            distance_non_connected.append(euclidean(reg.qubits[f"q{i}"], reg.qubits[f"q{j}"]))

Omega = device.interaction_coeff / np.min(distance_non_connected) ** 6 * 10
delta_0 = -Omega  # just has to be negative
delta_f = -delta_0  # just has to be positive
T = 40000  # time in ns, we choose a time long enough to ensure the propagation of information in the system

#add global Rydberg pulse on top of the local detuning map to vary blockade radii

adiabatic_pulse = pulser.Pulse(
    pulser.InterpolatedWaveform(T, [1e-9, Omega, 1e-9]),
    pulser.InterpolatedWaveform(T, [delta_0, 0, delta_f]),
    0,
)
sequence.add(adiabatic_pulse, "rydberg_global")

# Constant pulse added to the DMM

sequence.add_dmm_detuning(pulser.ConstantWaveform(T, -delta_f), "dmm_0")

#visualize pulse sequences

if test_plotting:

  sequence.draw(draw_detuning_maps=True,draw_qubit_det=True,draw_qubit_amp=True,)  # ,fig_name= "no_final_amplitude.pdf"


In [ ]:
simul = pulser_simulation.QutipBackendV2(sequence)
results = simul.run()
count_dict = results.final_bitstrings

if test_plotting:

  #if I am interpreting this correctly, it is demonstrating the collection of multiple hypotheses

  plot_distribution(count_dict)